In [3]:
import pandas as pd
import numpy as np
# from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score)
from sklearn.model_selection import RandomizedSearchCV

In [4]:
# Load data
df = pd.read_csv('tournament_model_ml.csv')
#df = pd.read_csv('data/tournament_model_adv_ml.csv')

In [5]:
# Features and target
features = ['3P%_diff', 'AST_diff', 'FG%_diff', 'FT%_diff', 'SRS_diff',
            'TOV_diff', 'TRB_diff', 'seed_diff', 'win_pct_diff']

# features = [
#     'SRS_diff', 'win_pct_diff', 'eFG%_diff', 'TS%_diff', 'TRB%_diff', 'AST%_diff', 'TOV%_diff', 'ORtg_diff'
# ]

            # Train/test split by season
train = df[df['season'] <= 2022]
test  = df[df['season'] >= 2023]

X_train, y_train = train[features], train['win_label']
X_test,  y_test  = test[features],  test['win_label']

print(f"Train rows: {len(train)} | Test rows: {len(test)}")

Train rows: 754 | Test rows: 314


In [6]:
from scipy.stats import randint
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV



param_dist = {
    'n_estimators':      (100,800),
    'max_depth': [None] + list(range(5, 31)),
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 8),
    'max_features':      ['sqrt', 'log2', None],
    'bootstrap':         [True, False]
}

rf = RandomForestClassifier(random_state=42, n_jobs=-1)


# Randomized search — tries 50 random combinations, 5-fold CV
search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=50,           # number of random combos to try
    cv=5,                # 5-fold cross-validation
    scoring='accuracy',   # optimize for accuracy
    random_state=42,
    n_jobs=-1,
    verbose=1
)


In [7]:
search.fit(X_train, y_train)

print(f"\nBest Parameters: {search.best_params_}")
print(f"Best CV AUC:     {search.best_score_:.4f}")

# Evaluate best model on test set
best_rf = search.best_estimator_

y_pred  = best_rf.predict(X_test)
y_proba = best_rf.predict_proba(X_test)[:, 1]

print(f"\nTest Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Test ROC-AUC:   {roc_auc_score(y_test, y_proba):.4f}")
print(f"\nConfusion Matrix:\n{confusion_matrix(y_test, y_pred)}")
print(f"\nClassification Report:\n{classification_report(y_test, y_pred)}")

# Feature importances from best model
importances = sorted(zip(features, best_rf.feature_importances_),
                     key=lambda x: x[1], reverse=True)
print("\nFeature Importances:")
for feat, imp in importances:
    print(f"  {feat:<15} {imp:.4f}")

Fitting 5 folds for each of 50 candidates, totalling 250 fits

Best Parameters: {'bootstrap': True, 'max_depth': 19, 'max_features': None, 'min_samples_leaf': 6, 'min_samples_split': 9, 'n_estimators': 100}
Best CV AUC:     0.7480

Test Accuracy:  0.7675
Test ROC-AUC:   0.8419

Confusion Matrix:
[[121  36]
 [ 37 120]]

Classification Report:
              precision    recall  f1-score   support

           0       0.77      0.77      0.77       157
           1       0.77      0.76      0.77       157

    accuracy                           0.77       314
   macro avg       0.77      0.77      0.77       314
weighted avg       0.77      0.77      0.77       314


Feature Importances:
  SRS_diff        0.5042
  TRB_diff        0.0932
  AST_diff        0.0883
  win_pct_diff    0.0672
  FT%_diff        0.0614
  FG%_diff        0.0536
  3P%_diff        0.0467
  TOV_diff        0.0463
  seed_diff       0.0391


In [8]:
# # ADABOOST MODEL

# # Parameter grid for AdaBoost
# ada_param_dist = {
#     'n_estimators':  [50, 100, 200, 300, 500],
#     'learning_rate': [0.01, 0.1, 0.5, 1.0, 1.5]
# }

# ada = AdaBoostClassifier(random_state=42)

# # Randomized search for AdaBoost
# ada_search = RandomizedSearchCV(
#     estimator=ada,
#     param_distributions=ada_param_dist,
#     n_iter=20,           # fewer iterations since AdaBoost is faster
#     cv=5,
#     scoring='roc_auc',
#     random_state=42,
#     n_jobs=-1,
#     verbose=1
# )

# ada_search.fit(X_train, y_train)

# print(f"\nAdaBoost Best Parameters: {ada_search.best_params_}")
# print(f"AdaBoost Best CV AUC:     {ada_search.best_score_:.4f}")

# # Evaluate best AdaBoost model on test set
# best_ada = ada_search.best_estimator_

# ada_y_pred  = best_ada.predict(X_test)
# ada_y_proba = best_ada.predict_proba(X_test)[:, 1]

# print(f"\nAdaBoost Test Accuracy:  {accuracy_score(y_test, ada_y_pred):.4f}")
# print(f"AdaBoost Test ROC-AUC:   {roc_auc_score(y_test, ada_y_proba):.4f}")
# print(f"\nAdaBoost Confusion Matrix:\n{confusion_matrix(y_test, ada_y_pred)}")
# print(f"\nAdaBoost Classification Report:\n{classification_report(y_test, ada_y_pred)}")

# # Feature importances from best AdaBoost model
# ada_importances = sorted(zip(features, best_ada.feature_importances_),
#                         key=lambda x: x[1], reverse=True)
# print("\nAdaBoost Feature Importances:")
# for feat, imp in ada_importances:
#     print(f"  {feat:<15} {imp:.4f}")